In [2]:
import pandas as pd
import numpy as np

train_ratings = pd.read_csv(
    "../datasets/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../datasets/processed/test_ratings.csv"
)

recipes = pd.read_csv(
    "../../datasets/RAW_recipes.csv"
)

print("Train shape:", train_ratings.shape)
print("Test shape:", test_ratings.shape)
print("Recipes shape:", recipes.shape)

Train shape: (428067, 5)
Test shape: (106980, 5)
Recipes shape: (231637, 12)


In [3]:
num_users = train_ratings["user_id"].nunique()
num_recipes = train_ratings["recipe_id"].nunique()

print("Users in train:", num_users)
print("Recipes in train:", num_recipes)
print("Ratings in train:", len(train_ratings))

Users in train: 17034
Recipes in train: 40027
Ratings in train: 428067


In [4]:
user_ids = train_ratings["user_id"].unique()
recipe_ids = train_ratings["recipe_id"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

recipe_to_index = {
    recipe_id: index
    for index, recipe_id in enumerate(recipe_ids)
}

print("Number of users:", len(user_to_index))
print("Number of recipes:", len(recipe_to_index))

Number of users: 17034
Number of recipes: 40027


In [5]:
from scipy.sparse import csr_matrix

rows = train_ratings["user_id"].map(user_to_index)
cols = train_ratings["recipe_id"].map(recipe_to_index)
values = train_ratings["rating"]

user_item_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_to_index), len(recipe_to_index))
)

print("User-item matrix shape:", user_item_matrix.shape)
print("Number of stored ratings:", user_item_matrix.nnz)

User-item matrix shape: (17034, 40027)
Number of stored ratings: 428067


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

test_user_id = test_ratings["user_id"].iloc[0]

user_index = user_to_index[test_user_id]

user_vector = user_item_matrix[user_index]

similarities = cosine_similarity(
    user_vector,
    user_item_matrix
).flatten()

similarities[user_index] = -1

similar_users_indices = np.argsort(similarities)[::-1][:10]

print("Test user:", test_user_id)
print("Most similar users:")
print(similar_users_indices)

Test user: 162826
Most similar users:
[ 4464  9276 15624 16129 13439  9220 12042 10517  6121 16725]


In [7]:
similar_users = pd.DataFrame({
    "user_id": [user_ids[i] for i in similar_users_indices],
    "similarity": [similarities[i] for i in similar_users_indices]
})

similar_users

,user_id,similarity
0,452437,0.079706
1,36670,0.060448
2,1165678,0.056764
3,571076,0.056194
4,204122,0.056194
5,985766,0.055221
6,185146,0.054536
7,226581,0.052175
8,188637,0.051910
9,508497,0.050261


In [8]:
# Recepti koje je test korisnik već ocenio
user_seen_recipes = set(
    train_ratings[
        train_ratings["user_id"] == test_user_id
        ]["recipe_id"]
)

# Uzmi interakcije sličnih korisnika
similar_user_ids = similar_users["user_id"].tolist()

candidate_ratings = train_ratings[
    train_ratings["user_id"].isin(similar_user_ids)
].copy()

# Izbaci recepte koje je naš korisnik već video
candidate_ratings = candidate_ratings[
    ~candidate_ratings["recipe_id"].isin(user_seen_recipes)
].copy()

print("Recipes already seen by test user:", len(user_seen_recipes))
print("Candidate interactions:", len(candidate_ratings))
print("Candidate recipes:", candidate_ratings["recipe_id"].nunique())

Recipes already seen by test user: 323
Candidate interactions: 54
Candidate recipes: 54


In [9]:
# Dodaj similarity svakom user-u
candidate_ratings = candidate_ratings.merge(
    similar_users,
    on="user_id",
    how="left"
)

# Weighted score:
# ocena * similarity korisnika
candidate_ratings["weighted_rating"] = (
        candidate_ratings["rating"] * candidate_ratings["similarity"]
)

# Zbirni score za svaki recept
recipe_scores = (
    candidate_ratings
    .groupby("recipe_id")
    .agg(
        score=("weighted_rating", "sum"),
        supporting_users=("user_id", "nunique")
    )
    .reset_index()
    .sort_values("score", ascending=False)
)

print("Number of candidate recipes:", len(recipe_scores))

recipe_scores.head(10)

Number of candidate recipes: 54


,recipe_id,score,supporting_users
23,140150,0.398532,1
17,95222,0.280970,1
28,155686,0.280970,1
51,327703,0.280970,1
18,104803,0.280970,1
5,29100,0.276104,1
16,88453,0.276104,1
20,105417,0.276104,1
12,56708,0.276104,1
13,57629,0.276104,1


In [10]:
top_10_recommendations = (
    recipe_scores
    .head(10)
    .merge(
        recipes[["id", "name"]],
        left_on="recipe_id",
        right_on="id",
        how="left"
    )
    [["recipe_id", "name", "score", "supporting_users"]]
)

top_10_recommendations

,recipe_id,name,score,supporting_users
0,140150,easy and delicious meringue cookies,0.398532,1
1,95222,pork chops yum yum,0.280970,1
2,155686,crusty macaroni and cheese,0.280970,1
3,327703,old fashioned luncheonette hot dog,0.280970,1
4,104803,southern style cabbage,0.280970,1
5,29100,ciabatta italian slipper bread,0.276104,1
6,88453,best ever bbq chicken,0.276104,1
7,105417,ruth s homemade vanilla extract,0.276104,1
8,56708,chinese style chicken thighs,0.276104,1
9,57629,erna s apple pie muffins,0.276104,1


In [11]:
def recommend_for_user(user_id, k=10, n_similar_users=10):
    # Proveri da li korisnik postoji u trening skupu
    if user_id not in user_to_index:
        return pd.DataFrame(columns=["recipe_id", "name", "score", "supporting_users"])

    # Indeks korisnika u user-item matrici
    user_index = user_to_index[user_id]

    # Sličnost sa svim korisnicima
    user_similarities = cosine_similarity(
        user_item_matrix[user_index],
        user_item_matrix
    ).flatten()

    # Izbaci samog korisnika
    user_similarities[user_index] = -1

    # Indeksi najsličnijih korisnika
    similar_indices = np.argsort(user_similarities)[-n_similar_users:][::-1]

    # Tabela sličnih korisnika
    similar_users_df = pd.DataFrame({
        "user_id": [user_ids[i] for i in similar_indices],
        "similarity": [user_similarities[i] for i in similar_indices]
    })

    # Recepti koje je korisnik već ocenio
    seen_recipes = set(
        train_ratings[
            train_ratings["user_id"] == user_id
            ]["recipe_id"]
    )

    # Interakcije sličnih korisnika
    candidates = train_ratings[
        train_ratings["user_id"].isin(similar_users_df["user_id"])
    ].copy()

    # Izbaci već viđene recepte
    candidates = candidates[
        ~candidates["recipe_id"].isin(seen_recipes)
    ].copy()

    # Dodaj sličnost korisnika
    candidates = candidates.merge(
        similar_users_df,
        on="user_id",
        how="left"
    )

    # Weighted score
    candidates["weighted_rating"] = (
            candidates["rating"] * candidates["similarity"]
    )

    # Rangiranje recepata
    scores = (
        candidates
        .groupby("recipe_id")
        .agg(
            score=("weighted_rating", "sum"),
            supporting_users=("user_id", "nunique")
        )
        .reset_index()
        .sort_values(
            ["score", "supporting_users"],
            ascending=[False, False]
        )
        .head(k)
    )

    # Dodaj naziv recepta
    recommendations = scores.merge(
        recipes[["id", "name"]],
        left_on="recipe_id",
        right_on="id",
        how="left"
    )

    return recommendations[
        ["recipe_id", "name", "score", "supporting_users"]
    ]

In [12]:
recommendations = recommend_for_user(test_user_id, k=10)

recommendations

,recipe_id,name,score,supporting_users
0,140150,easy and delicious meringue cookies,0.398532,1
1,95222,pork chops yum yum,0.280970,1
2,104803,southern style cabbage,0.280970,1
3,155686,crusty macaroni and cheese,0.280970,1
4,327703,old fashioned luncheonette hot dog,0.280970,1
5,749,zucchini lasagna lasagne low carb,0.276104,1
6,24344,delicious corn muffins,0.276104,1
7,29100,ciabatta italian slipper bread,0.276104,1
8,30484,chicago italian beef,0.276104,1
9,56708,chinese style chicken thighs,0.276104,1


In [13]:
def evaluate_collaborative_filtering(k=10, max_users=1000):
    precisions = []

    # Korisnici koji postoje u test skupu
    test_users = test_ratings["user_id"].unique()

    # Ograničavamo broj korisnika zbog vremena izvršavanja
    test_users = test_users[:max_users]

    for user_id in test_users:
        recommendations = recommend_for_user(
            user_id,
            k=k,
            n_similar_users=10
        )

        if recommendations.empty:
            continue

        recommended_recipes = set(
            recommendations["recipe_id"]
        )

        actual_recipes = set(
            test_ratings[
                test_ratings["user_id"] == user_id
                ]["recipe_id"]
        )

        hits = len(recommended_recipes & actual_recipes)

        precisions.append(hits / k)

    if len(precisions) == 0:
        return 0.0

    return np.mean(precisions)

In [14]:
cf_precision_at_10 = evaluate_collaborative_filtering(
    k=10,
    max_users=1000
)

print("Collaborative Filtering Precision@10:", cf_precision_at_10)

Collaborative Filtering Precision@10: 0.0187


In [15]:
def evaluate_collaborative_filtering_recall(k=10, max_users=1000):
    recalls = []

    test_users = test_ratings["user_id"].unique()
    test_users = test_users[:max_users]

    for user_id in test_users:
        recommendations = recommend_for_user(
            user_id,
            k=k,
            n_similar_users=10
        )

        if recommendations.empty:
            continue

        recommended_recipes = set(
            recommendations["recipe_id"]
        )

        actual_recipes = set(
            test_ratings[
                test_ratings["user_id"] == user_id
                ]["recipe_id"]
        )

        if len(actual_recipes) == 0:
            continue

        hits = len(recommended_recipes & actual_recipes)

        recalls.append(hits / len(actual_recipes))

    if len(recalls) == 0:
        return 0.0

    return np.mean(recalls)

In [16]:
cf_recall_at_10 = evaluate_collaborative_filtering_recall(
    k=10,
    max_users=1000
)

print("Collaborative Filtering Recall@10:", cf_recall_at_10)

Collaborative Filtering Recall@10: 0.00670987353118126


In [17]:
import pickle
import os

cf_model = {
    "user_item_matrix": user_item_matrix,
    "user_ids": user_ids,
    "user_to_index": user_to_index,
    "recipe_ids": recipe_ids,
    "n_similar_users": 10,
    "k": 10,
    "precision_at_10": cf_precision_at_10,
    "recall_at_10": cf_recall_at_10
}

model_path = "../saved_models/collaborative_filtering.pkl"

with open(model_path, "wb") as f:
    pickle.dump(cf_model, f)

print("Collaborative Filtering model saved to:")
print(os.path.abspath(model_path))

Collaborative Filtering model saved to:
C:\Users\Nikola\Desktop\FoodRecommendationSystem\ai\saved_models\collaborative_filtering.pkl
